# Statistical Significance Testing (v7), notebook version v4

For each city x horizon: fold-level MASE differences of two models are averaged into 35 weekly differences, then tested.
- p-value: Diebold-Mariano style t-test with Newey-West (HAC) variance on the weekly differences
- 95% CI: paired bootstrap over weeks
- Holm correction within each claim group: baselines / ablation / robustness / reality check; each comparison appears once
- `mean_diff` = MASE(A) - MASE(B); negative = A is better
- NEW in v4: two difference-in-differences tests (negative = TabPFN better):
  - ablation: (TabPFN - TabPFN_NoWeather) - (XGBoost - XGBoost_NoWeather), i.e. does weather help TabPFN more than XGBoost?
  - robustness: (TabPFN[degraded] - TabPFN[clean]) - (XGBoost[degraded] - XGBoost[clean]), i.e. does degraded weather hurt TabPFN more than XGBoost?
  - They add 12 tests each to the ablation and robustness groups, so the Holm p-values in those groups change.
- `borderline` (changed in v4) = the raw t-test (p < 0.05) and the bootstrap CI (excludes 0) disagree. Both are at the same unadjusted 5 % level. Holm significance is a separate column.

Everything is saved as CSV in `../results_v7/tables/cross_city/` (tests, summary, weekly differences).
Comparisons and open decisions: first code cell. Last cell: are neighbouring weeks related (check for the bootstrap)?


In [1]:
print("Notebook version v4, 11 Oct 2026: + difference-in-differences tests, borderline = raw p vs CI, week-correlation check")
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import stats
from statsmodels.stats.multitest import multipletests

RESULTS_VERSION = "v7"
DATASETS = ["seoul", "london", "washington"]
FOLD_KEY = ["dataset", "model", "horizon", "weather_scenario", "fold"]
N_BOOT, SEED = 10_000, 42

# ---- decisions still open (edit here only) ---------------------------------------------------
INCLUDE_XGB_NOWEATHER = True            # like-for-like baselines (same inputs), XGBoost ablation, degraded vs NoWeather
INCLUDE_TABPFN_VS_XGB_DEGRADED = True   # is TabPFN still ahead of XGBoost once the weather is degraded?
INCLUDE_DID = True                      # does weather help TabPFN MORE than XGBoost / does degradation hurt it MORE?

# (family, A model, A scenario, B model, B scenario); scenario: 'clean' or 'degraded'
# Every comparison appears ONCE, so it gets one Holm result.
COMPARISONS = [
    # Baselines (all clean weather): RQ1. TabPFN_NoWeather vs XGBoost is left out: different inputs.
    ("1_baselines", "TabPFN",           "clean", "XGBoost", "clean"),
    # Ablation: RQ2
    ("2_ablation",  "TabPFN",           "clean", "TabPFN_NoWeather", "clean"),
    # Robustness: RQ3 (clean vs degraded)
    ("3_robustness", "TabPFN",  "clean", "TabPFN",  "degraded"),
    ("3_robustness", "XGBoost", "clean", "XGBoost", "degraded"),
    # Reality check: degraded weather vs no weather at all
    ("4_reality",   "TabPFN",           "degraded", "TabPFN_NoWeather", "clean"),
]
if INCLUDE_XGB_NOWEATHER:
    COMPARISONS += [
        ("1_baselines", "TabPFN_NoWeather", "clean", "XGBoost_NoWeather", "clean"),
        ("2_ablation",  "XGBoost", "clean", "XGBoost_NoWeather", "clean"),
        ("4_reality",   "XGBoost", "degraded", "XGBoost_NoWeather", "clean"),
    ]
if INCLUDE_TABPFN_VS_XGB_DEGRADED:
    COMPARISONS += [("4_reality", "TabPFN", "degraded", "XGBoost", "degraded")]

# Difference-in-differences: per fold d = sum(coef * MASE) over the terms; negative = TabPFN better.
# (family, label, [(model, scenario, coef), ...])
DID_COMPARISONS = []
if INCLUDE_DID:
    DID_COMPARISONS += [
        ("2_ablation",
         "DiD gain: (TabPFN - TabPFN_NoWeather) - (XGBoost - XGBoost_NoWeather) [clean]",
         [("TabPFN", "clean", 1), ("TabPFN_NoWeather", "clean", -1),
          ("XGBoost", "clean", -1), ("XGBoost_NoWeather", "clean", 1)]),
        ("3_robustness",
         "DiD degradation: (TabPFN[degraded] - TabPFN[clean]) - (XGBoost[degraded] - XGBoost[clean])",
         [("TabPFN", "degraded", 1), ("TabPFN", "clean", -1),
          ("XGBoost", "degraded", -1), ("XGBoost", "clean", 1)]),
    ]

SC = {"clean": "clean_only", "degraded": "degraded"}


def all_comparisons():
    """Every test as (family, label, terms); a plain A vs B comparison is A*1 + B*(-1)."""
    out = [(fam, f"{mA}[{sA}] vs {mB}[{sB}]", [(mA, sA, 1), (mB, sB, -1)])
           for fam, mA, sA, mB, sB in COMPARISONS]
    return out + DID_COMPARISONS


def load():
    df = pd.read_csv(f"../results/detailed_results_master_{RESULTS_VERSION}.csv")
    df = df[df["version"] == RESULTS_VERSION]
    df = df.sort_values("timestamp").drop_duplicates(FOLD_KEY, keep="last")
    return df[df["dataset"].isin(DATASETS)]


Notebook version v4, 11 Oct 2026: + difference-in-differences tests, borderline = raw p vs CI, week-correlation check


In [2]:
def weekly_diffs(df, terms):
    """Weekly mean of d = sum(coef * MASE) over the (model, scenario, coef) terms.
    Folds are matched across all terms. Fold -> week by the fold's start hour (fold*horizon // 168)."""
    keys = ["dataset", "horizon", "fold"]
    m = None
    for i, (mod, s, _) in enumerate(terms):
        x = df[(df["model"] == mod) & (df["weather_scenario"] == SC[s])][keys + ["MASE"]]
        x = x.rename(columns={"MASE": f"MASE_{i}"})
        m = x if m is None else m.merge(x, on=keys)
    if m is None or m.empty:
        raise ValueError(f"No matching folds for {terms}")
    first = m.groupby(["dataset", "horizon"])["fold"].transform("min")   # works for 0- or 1-based folds
    m["week"] = (m["fold"] - first) * m["horizon"] // 168
    m["d"] = sum(coef * m[f"MASE_{i}"] for i, (_, _, coef) in enumerate(terms))
    dropped = m["d"].isna().groupby([m["dataset"], m["horizon"]]).sum()  # folds with undefined MASE
    m = m.dropna(subset=["d"])
    wk = m.groupby(["dataset", "horizon", "week"])["d"].agg(d="mean", n_folds="size").reset_index()
    return wk, dropped


def dm_hac(d):
    """t-test of mean(d)=0 with Newey-West (Bartlett) variance; t distribution with n-1 df."""
    n = len(d)
    if n < 5 or np.allclose(d, d[0]):
        return np.nan, np.nan
    e = d - d.mean()
    L = int(np.floor(n ** (1 / 3)))
    v = (e @ e) / n
    for k in range(1, L + 1):
        v += 2 * (1 - k / (L + 1)) * (e[k:] @ e[:-k]) / n
    if v <= 0:
        return np.nan, np.nan
    t = d.mean() / np.sqrt(v / n)
    return t, 2 * stats.t.sf(abs(t), n - 1)


def boot_ci(d, rng):
    idx = rng.integers(0, len(d), size=(N_BOOT, len(d)))
    return np.percentile(d[idx].mean(axis=1), [2.5, 97.5])


def main():
    df, rng, rows, wk_all = load(), np.random.default_rng(SEED), [], []
    for fam, label, terms in all_comparisons():
        wk, dropped = weekly_diffs(df, terms)
        wk_all.append(wk.assign(family=fam, comparison=label))
        for (ds, h), g in wk.groupby(["dataset", "horizon"]):
            d = g["d"].to_numpy()
            t, p = dm_hac(d)
            lo, hi = boot_ci(d, rng)
            rows.append(dict(family=fam, comparison=label, dataset=ds, horizon=h,
                             n_weeks=len(d), n_folds_used=int(g["n_folds"].sum()),
                             n_folds_dropped=int(dropped.get((ds, h), 0)), mean_diff=d.mean(), ci_low=lo, ci_high=hi, t=t, p=p))
    res = pd.DataFrame(rows)
    res["p_holm"], res["significant"] = np.nan, False
    for fam, idx in res.groupby("family").groups.items():
        ok = res.loc[idx, "p"].notna()
        if ok.any():
            rej, padj, _, _ = multipletests(res.loc[idx[ok], "p"], method="holm")
            res.loc[idx[ok], "p_holm"], res.loc[idx[ok], "significant"] = padj, rej
    # borderline: the two checks of the SAME question at the SAME unadjusted 5 % level disagree
    # (raw t-test p < 0.05 vs bootstrap CI excluding 0). Holm significance is reported separately;
    # comparing it with the unadjusted CI would only flag the multiplicity correction.
    res["ci_excludes_zero"] = (res["ci_low"] > 0) | (res["ci_high"] < 0)
    res["borderline"] = res["ci_excludes_zero"] != (res["p"] < 0.05)
    res["better"] = np.where(~res["significant"], "n.s.", np.where(res["mean_diff"] < 0, "A", "B"))

    summ = (res.groupby(["family", "comparison"])
              .agg(tests=("p", "size"), sig=("significant", "sum"),
                   A_better=("better", lambda s: (s == "A").sum()),
                   B_better=("better", lambda s: (s == "B").sum()),
                   borderline=("borderline", "sum"),
                   weeks_min=("n_weeks", "min")).reset_index())
    out = Path(f"../results_{RESULTS_VERSION}/tables/cross_city")
    out.mkdir(parents=True, exist_ok=True)
    res.to_csv(out / "significance_v7_tests.csv", index=False)
    summ.to_csv(out / "significance_v7_summary.csv", index=False)
    pd.concat(wk_all, ignore_index=True).to_csv(out / "significance_v7_weekly_diffs.csv", index=False)
    print(f"Saved 3 files to {out}: significance_v7_tests.csv, significance_v7_summary.csv, significance_v7_weekly_diffs.csv")
    return res, summ


res, summ = main()
print(summ.to_string(index=False))


Saved 3 files to ../results_v7/tables/cross_city: significance_v7_tests.csv, significance_v7_summary.csv, significance_v7_weekly_diffs.csv
      family                                                                                 comparison  tests  sig  A_better  B_better  borderline  weeks_min
 1_baselines                                                            TabPFN[clean] vs XGBoost[clean]     12    6         6         0           1         35
 1_baselines                                        TabPFN_NoWeather[clean] vs XGBoost_NoWeather[clean]     12    3         3         0           1         35
  2_ablation              DiD gain: (TabPFN - TabPFN_NoWeather) - (XGBoost - XGBoost_NoWeather) [clean]     12   10        10         0           0         35
  2_ablation                                                   TabPFN[clean] vs TabPFN_NoWeather[clean]     12   12        12         0           0         35
  2_ablation                                                 XGBoo

In [3]:
# Optional: look at one claim group on screen (everything is already saved as CSV by main())
FAMILY_TO_SHOW = "4_reality"
cols = ["comparison", "dataset", "horizon", "n_weeks", "mean_diff", "ci_low", "ci_high", "p", "p_holm", "significant", "borderline"]
print(res.loc[res["family"] == FAMILY_TO_SHOW, cols].round(4).to_string(index=False))


                                   comparison    dataset  horizon  n_weeks  mean_diff  ci_low  ci_high      p  p_holm  significant  borderline
  TabPFN[degraded] vs TabPFN_NoWeather[clean]     london        6       35    -0.0230 -0.0466  -0.0001 0.0512  0.3608        False        True
  TabPFN[degraded] vs TabPFN_NoWeather[clean]     london       24       35    -0.0900 -0.1239  -0.0576 0.0000  0.0002         True       False
  TabPFN[degraded] vs TabPFN_NoWeather[clean]     london       48       35    -0.1020 -0.1542  -0.0503 0.0000  0.0009         True       False
  TabPFN[degraded] vs TabPFN_NoWeather[clean]     london      168       35    -0.1140 -0.1734  -0.0620 0.0006  0.0113         True       False
  TabPFN[degraded] vs TabPFN_NoWeather[clean]      seoul        6       35    -0.0963 -0.1215  -0.0714 0.0000  0.0000         True       False
  TabPFN[degraded] vs TabPFN_NoWeather[clean]      seoul       24       35    -0.2743 -0.3525  -0.2019 0.0000  0.0000         True       False

In [4]:
# ── Are neighbouring weeks related? (check for the bootstrap CIs) ─────────────
# The t-test allows a week's difference to be related to the next 1-3 weeks (Newey-West).
# The bootstrap CIs here, and the block bootstrap of skill score / win rate in the main
# notebook, draw weeks independently: they assume there is NO such relation.
# This cell measures it: correlation of each week's difference with the week 1, 2, 3 later,
# for every comparison x city x horizon (35 weeks each).
# With unrelated weeks, a correlation above +0.33 happens by chance in about 2.5 % of series.
wk = pd.read_csv(f"../results_{RESULTS_VERSION}/tables/cross_city/significance_v7_weekly_diffs.csv")
LIMIT = 1.96 / np.sqrt(35)
rows = []
for (comp, ds, h), g in wk.groupby(["comparison", "dataset", "horizon"]):
    s = g.sort_values("week")["d"].reset_index(drop=True)
    rows.append(dict(comparison=comp, dataset=ds, horizon=h,
                     **{f"lag{k}": s.autocorr(k) for k in (1, 2, 3)}))
ac = pd.DataFrame(rows)

print(f"{len(ac)} series of weekly differences | 'clearly related' = correlation above +{LIMIT:.2f}")
for k in (1, 2, 3):
    c = ac[f"lag{k}"].dropna()
    print(f"  {k} week(s) apart: median {c.median():+.2f} | share above +{LIMIT:.2f}: {(c > LIMIT).mean():.0%}  (chance: ~2.5%)")
print("\nMedian correlation with the next week, by horizon:")
print(ac.groupby("horizon")["lag1"].median().round(2).to_string())


132 series of weekly differences | 'clearly related' = correlation above +0.33
  1 week(s) apart: median +0.03 | share above +0.33: 5%  (chance: ~2.5%)
  2 week(s) apart: median -0.03 | share above +0.33: 0%  (chance: ~2.5%)
  3 week(s) apart: median -0.03 | share above +0.33: 3%  (chance: ~2.5%)

Median correlation with the next week, by horizon:
horizon
6      0.07
24     0.02
48    -0.00
168    0.05
